In [0]:
# ============================================
# DATA QUALITY TESTING
# ============================================

from pyspark.sql.functions import col, count

print("========== DATA QUALITY TESTING ==========\n")


# ------------------------------------------------
# 1. DUPLICATE TESTS
# ------------------------------------------------

print("1. DUPLICATE TESTS")

tests = {
    "orders": "order_id",
    "returns": "return_id",
    "refunds": "refund_id",
    "customers": "customer_id",
    "products": "product_id",
    "sellers": "seller_id"
}

for table_name, key_column in tests.items():

    df = spark.table(f"ecommerce_returns.silver.{table_name}")

    duplicate_count = (
        df.groupBy(key_column)
          .count()
          .filter(col("count") > 1)
          .count()
    )

    if duplicate_count == 0:
        print(f"PASS - {table_name}: no duplicate {key_column}")
    else:
        print(f"FAIL - {table_name}: {duplicate_count} duplicate keys")


# ------------------------------------------------
# 2. NULL / MISSING KEY TESTS
# ------------------------------------------------

print("\n2. NULL KEY TESTS")

for table_name, key_column in tests.items():

    df = spark.table(f"ecommerce_returns.silver.{table_name}")

    null_count = df.filter(col(key_column).isNull()).count()

    if null_count == 0:
        print(f"PASS - {table_name}: no null {key_column}")
    else:
        print(f"FAIL - {table_name}: {null_count} null keys")


# ------------------------------------------------
# 3. ORDER QUANTITY TEST
# ------------------------------------------------

print("\n3. ORDER QUANTITY TEST")

orders = spark.table("ecommerce_returns.silver.orders")

invalid_quantity = orders.filter(
    col("quantity") <= 0
).count()

if invalid_quantity == 0:
    print("PASS - Orders contain no invalid quantities")
else:
    print(f"FAIL - {invalid_quantity} orders have quantity <= 0")


# ------------------------------------------------
# 4. RETURN DATE TEST
# ------------------------------------------------

print("\n4. RETURN DATE TEST")

returns = spark.table("ecommerce_returns.silver.returns")

orders = spark.table("ecommerce_returns.silver.orders")

invalid_return_dates = (
    returns.alias("r")
    .join(
        orders.select("order_id", "order_date").alias("o"),
        col("r.order_id") == col("o.order_id"),
        "left"
    )
    .filter(col("r.return_date") < col("o.order_date"))
    .count()
)

if invalid_return_dates == 0:
    print("PASS - No return occurs before order date")
else:
    print(f"FAIL - {invalid_return_dates} invalid return dates")


# ------------------------------------------------
# 5. ORPHAN RETURN TEST
# ------------------------------------------------

print("\n5. ORPHAN RETURN TEST")

orphan_returns = (
    returns.alias("r")
    .join(
        orders.select("order_id").alias("o"),
        col("r.order_id") == col("o.order_id"),
        "left_anti"
    )
    .count()
)

if orphan_returns == 0:
    print("PASS - No orphan returns")
else:
    print(f"FAIL - {orphan_returns} orphan returns")


# ------------------------------------------------
# 6. ORPHAN REFUND TEST
# ------------------------------------------------

print("\n6. ORPHAN REFUND TEST")

refunds = spark.table("ecommerce_returns.silver.refunds")

orphan_refunds = (
    refunds.alias("f")
    .join(
        returns.select("return_id").alias("r"),
        col("f.return_id") == col("r.return_id"),
        "left_anti"
    )
    .count()
)

if orphan_refunds == 0:
    print("PASS - No orphan refunds")
else:
    print(f"FAIL - {orphan_refunds} orphan refunds")


# ------------------------------------------------
# 7. REFUND AMOUNT TEST
# ------------------------------------------------

print("\n7. REFUND AMOUNT TEST")

invalid_refunds = refunds.filter(
    col("refund_amount") < 0
).count()

if invalid_refunds == 0:
    print("PASS - No negative refund amounts")
else:
    print(f"FAIL - {invalid_refunds} negative refund amounts")


# ------------------------------------------------
# 8. GOLD TABLE EXISTENCE TEST
# ------------------------------------------------

print("\n8. GOLD TABLE TEST")

gold_tables = [
    "dim_customer",
    "dim_product",
    "dim_seller",
    "dim_date",
    "fact_orders",
    "fact_returns",
    "fact_refunds",
    "product_return_metrics",
    "seller_return_metrics",
    "monthly_return_metrics",
    "refund_sla_metrics"
]

for table_name in gold_tables:

    try:
        df = spark.table(
            f"ecommerce_returns.gold.{table_name}"
        )

        row_count = df.count()

        if row_count > 0:
            print(f"PASS - gold.{table_name}: {row_count:,} rows")
        else:
            print(f"FAIL - gold.{table_name}: empty table")

    except Exception:
        print(f"FAIL - gold.{table_name}: table not found")


# ------------------------------------------------
# FINAL
# ------------------------------------------------

print("\n========== DATA QUALITY TESTING COMPLETE ==========")

In [0]:
# Investigate the 15 orphan refunds

refunds = spark.table("ecommerce_returns.silver.refunds")
returns = spark.table("ecommerce_returns.silver.returns")

orphan_refunds_df = (
    refunds.alias("f")
    .join(
        returns.select("return_id").alias("r"),
        col("f.return_id") == col("r.return_id"),
        "left_anti"
    )
)

display(orphan_refunds_df)

In [0]:
from pyspark.sql.functions import col

refunds = spark.table("ecommerce_returns.silver.refunds")
returns = spark.table("ecommerce_returns.silver.returns")

# Find refunds whose return_id does not exist in Silver returns
orphan_refunds = (
    refunds.alias("f")
    .join(
        returns.select("return_id").alias("r"),
        col("f.return_id") == col("r.return_id"),
        "left_anti"
    )
)

# Save invalid records to quarantine
orphan_refunds.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_returns.quarantine.refunds_rejected")

print(f"Quarantined refunds: {orphan_refunds.count()}")

In [0]:
valid_refunds = (
    refunds.alias("f")
    .join(
        returns.select("return_id").alias("r"),
        col("f.return_id") == col("r.return_id"),
        "inner"
    )
    .select("f.*")
)

valid_refunds.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_returns.silver.refunds")

print(f"Silver refunds after quarantine: {valid_refunds.count()}")